# 10 M3: ทดสอบย้อนหลังทีละไตรมาส และตรวจจับเหตุการณ์ผิดปกติ

Pain point ของ BART
- ก่อนจะใช้โมเดลพยากรณ์วางแผนจริง ต้องมั่นใจว่าแม่นสม่ำเสมอ ไม่ได้บังเอิญแม่นแค่ช่วงทดสอบช่วงเดียว
- ช่วง P90 ที่จะใช้กำหนดขบวนสำรอง ต้องครอบค่าจริงได้ตามที่บอก (M1 ช่วงเช้าครอบได้แค่ 68% จากเป้า 80%)
- วันที่คนล้นหรือคนหายผิดปกติ ต้องรู้ว่าเกิดที่ไหน บ่อยแค่ไหน เพื่อเตรียมรถและพนักงาน

notebook นี้มี 3 ส่วน
1. ทดสอบย้อนหลัง 8 ไตรมาส (ก.ค. 2021 ถึง 2 พ.ค. 2023) แต่ละไตรมาสเทรน M1 ใหม่ด้วยข้อมูลก่อนหน้าเท่านั้น
2. ขยายช่วง P10-P90 ด้วย error ของไตรมาสก่อนหน้า (conformal calibration) แล้ววัดว่าครอบค่าจริงได้ใกล้ 80% ขึ้นไหม
3. ใช้ค่าพยากรณ์ล่วงหน้าทั้ง 8 ไตรมาสหาวันที่ผิดปกติ ทั้งคนล้นและคนหาย

In [1]:
import sys
from pathlib import Path

if Path.cwd().name == "notebooks":
    ROOT = Path.cwd().parent
else:
    ROOT = Path.cwd()
sys.path.append(str(ROOT / "src"))

import gc
import json
import time
from datetime import date, timedelta

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import polars as pl
import xgboost as xgb

from bart import viz
from bart.config import FEATURES, MODELS, TAB
from bart.perf import stage

viz.notebook_mode()

## 1. ข้อมูลและค่าตั้งค่าจาก M1

- ใช้ feature ชุดเดียวกับ M1 (`m1_features.parquet`) และไม่ใช้ช่วง break COVID (2020-03-09 ถึง 2020-05-31) เทรนเหมือนเดิม
- ใช้ค่าที่จูนได้จาก M1 (`m1_best_params.json`) ไม่จูนใหม่ เพื่อทดสอบข้อเสนอของ M1 ว่าเทรนใหม่ทุกไตรมาสด้วยค่าเดิมก็พอ
- จำนวนต้นไม้คงที่เท่ากับโมเดลสุดท้ายของ M1 เพราะการทดสอบย้อนหลังไม่มี valid ให้ early stopping

In [2]:
feature_table = pl.read_parquet(FEATURES / "m1_features.parquet")

feature_columns = [
    "station", "corridor", "hour", "day_of_week", "month", "is_holiday", "holiday_name",
    "days_to_holiday", "days_since_holiday", "is_pride_day", "covid_period", "days_since_open",
    "lag_1w", "lag_2w", "lag_3w", "lag_4w", "lag_52w",
    "lag_4w_mean", "lag_4w_median", "lag_4w_std", "lag_4w_min", "lag_4w_max",
    "lag_1w_was_holiday", "station_mean_7d", "station_mean_28d", "station_trend",
    "system_lag_1w", "system_mean_7d",
]
categorical_columns = ["station", "corridor", "holiday_name", "covid_period"]
boolean_columns = ["is_holiday", "is_pride_day", "lag_1w_was_holiday"]

categories = {}
for column in categorical_columns:
    categories[column] = feature_table[column].unique().sort().to_list()

def to_model_input(frame):
    table = frame.select(feature_columns).to_pandas()
    for column in categorical_columns:
        table[column] = pd.Categorical(table[column], categories=categories[column])
    for column in boolean_columns:
        table[column] = table[column].astype("float32")
    return table

with open(MODELS / "m1_best_params.json") as handle:
    m1 = json.load(handle)

if m1["variant"] != "all":
    raise ValueError(f"this notebook expects the 'all' training window, M1 chose {m1['variant']}")

point_params = {
    "device": "cuda",
    "tree_method": "hist",
    "objective": "reg:tweedie",
    "eval_metric": "mae",
    "seed": 42,
}
for name in ["max_depth", "min_child_weight", "subsample", "colsample_bytree", "lambda", "alpha",
            "tweedie_variance_power"]:
    point_params[name] = m1["tuned"][name]
point_params["learning_rate"] = m1["final_learning_rate"]

quantile_params = dict(point_params)
del quantile_params["tweedie_variance_power"]
quantile_params["max_depth"] = min(point_params["max_depth"], 8)
quantile_params["learning_rate"] = 0.05
quantile_params["objective"] = "reg:quantileerror"
quantile_params["quantile_alpha"] = np.array([0.1, 0.5, 0.9])
quantile_params["eval_metric"] = "quantile"

POINT_ROUNDS = m1["point_rounds"]
QUANTILE_ROUNDS = m1["quantile_rounds"]

print(f"feature rows: {feature_table.height:,}")
print(f"point model   : {POINT_ROUNDS} trees, learning rate {point_params['learning_rate']:.4f}, depth {point_params['max_depth']}")
print(f"quantile model: {QUANTILE_ROUNDS} trees, learning rate {quantile_params['learning_rate']}, depth {quantile_params['max_depth']}")

feature rows: 5,000,232
point model   : 972 trees, learning rate 0.0123, depth 12
quantile model: 1575 trees, learning rate 0.05, depth 8


### ไตรมาสที่ทดสอบ

- พยากรณ์ออกล่วงหน้า 7 วัน ข้อมูลล่าสุดที่รู้ตอนออกพยากรณ์วันแรกของไตรมาสคือ 7 วันก่อนหน้า
- จึงเทรนด้วยวันที่ก่อน "วันแรกของไตรมาส - 7 วัน" เท่านั้น แล้วใช้โมเดลนั้นพยากรณ์ทั้งไตรมาส
- ไตรมาสสุดท้ายมีแค่ 1 เม.ย. ถึง 2 พ.ค. 2023 เพราะข้อมูลจบตรงนั้น

In [3]:
last_date = date.fromisoformat(str(feature_table["date"].max()))

quarter_starts = [
    date(2021, 7, 1), date(2021, 10, 1),
    date(2022, 1, 1), date(2022, 4, 1), date(2022, 7, 1), date(2022, 10, 1),
    date(2023, 1, 1), date(2023, 4, 1),
]


def quarter_label(start):
    quarter = (start.month - 1) // 3 + 1
    return f"{start.year}-Q{quarter}"


def quarter_end(start):
    if start.month == 10:
        next_start = date(start.year + 1, 1, 1)
    else:
        next_start = date(start.year, start.month + 3, 1)
    end = next_start - timedelta(days=1)
    if end > last_date:
        end = last_date
    return end


usable = feature_table.filter(pl.col("split") != "excluded")

folds = []
for start in quarter_starts:
    train_cut = start - timedelta(days=7)
    end = quarter_end(start)
    folds.append({
        "fold": quarter_label(start),
        "train_until": train_cut - timedelta(days=1),
        "forecast_from": start,
        "forecast_to": end,
        "train_rows": usable.filter(pl.col("date") < train_cut).height,
        "forecast_rows": usable.filter(pl.col("date").is_between(start, end)).height,
    })

fold_table = pd.DataFrame(folds).set_index("fold")
fold_table

,train_until,forecast_from,forecast_to,train_rows,forecast_rows
fold,,,,,
2021-Q3,2021-06-23,2021-07-01,2021-09-30,4091016,110400
2021-Q4,2021-09-23,2021-10-01,2021-12-31,4201416,110400
2022-Q1,2021-12-24,2022-01-01,2022-03-31,4311816,108000
2022-Q2,2022-03-24,2022-04-01,2022-06-30,4419816,109200
2022-Q3,2022-06-23,2022-07-01,2022-09-30,4529016,110400
2022-Q4,2022-09-23,2022-10-01,2022-12-31,4639416,110400
2023-Q1,2022-12-24,2023-01-01,2023-03-31,4749816,108000
2023-Q2,2023-03-24,2023-04-01,2023-05-02,4857816,38400


## 2. ทดสอบย้อนหลัง

แต่ละไตรมาสเทรน 2 โมเดลบน GPU แล้วพยากรณ์ไตรมาสนั้น
- โมเดลหลัก (Tweedie) ให้ค่าพยากรณ์
- โมเดล quantile ให้ P10, P50, P90

กันค้าง
- ผลของแต่ละไตรมาสบันทึกเป็น Parquet ทันทีที่เสร็จ ถ้ารันใหม่หรือหยุดกลางคัน จะข้ามไตรมาสที่มีไฟล์แล้ว
- แสดงเวลาและความคืบหน้าทุกไตรมาส
- ลบ matrix ออกจาก GPU หลังจบแต่ละไตรมาส

In [4]:
BACKTEST_DIR = MODELS / "m3_backtest"
BACKTEST_DIR.mkdir(parents=True, exist_ok=True)

KEEP_COLUMNS = ["station", "date", "hour", "day_of_week", "is_holiday", "target", "lag_4w_mean"]


def run_fold(fold):
    train_cut = fold["forecast_from"] - timedelta(days=7)
    train_rows = usable.filter(pl.col("date") < train_cut)
    forecast_rows = usable.filter(pl.col("date").is_between(fold["forecast_from"], fold["forecast_to"]))

    train_matrix = xgb.QuantileDMatrix(
        to_model_input(train_rows), train_rows["target"].to_numpy(), enable_categorical=True, max_bin=256,
    )
    forecast_matrix = xgb.QuantileDMatrix(
        to_model_input(forecast_rows), forecast_rows["target"].to_numpy(), ref=train_matrix,
        enable_categorical=True, max_bin=256,
    )

    point_model = xgb.train(point_params, train_matrix, num_boost_round=POINT_ROUNDS)
    quantile_model = xgb.train(quantile_params, train_matrix, num_boost_round=QUANTILE_ROUNDS)

    point = point_model.predict(forecast_matrix)
    bands = np.sort(quantile_model.predict(forecast_matrix), axis=1)

    result = forecast_rows.select(KEEP_COLUMNS).with_columns(
        pl.lit(fold["fold"]).alias("fold"),
        pl.Series("prediction", point),
        pl.Series("p10", bands[:, 0]),
        pl.Series("p50", bands[:, 1]),
        pl.Series("p90", bands[:, 2]),
    )

    del train_matrix, forecast_matrix, point_model, quantile_model
    gc.collect()
    return result


fold_seconds = {}
started_all = time.perf_counter()
with stage("10.backtest", gpu=True):
    for position, fold in enumerate(folds, start=1):
        path = BACKTEST_DIR / f"{fold['fold']}.parquet"
        if path.exists():
            print(f"{position}/{len(folds)} {fold['fold']}: already done, loaded from {path.name}")
            continue

        started = time.perf_counter()
        result = run_fold(fold)
        result.write_parquet(path)
        seconds = time.perf_counter() - started
        fold_seconds[fold["fold"]] = round(seconds)

        elapsed = (time.perf_counter() - started_all) / 60
        print(f"{position}/{len(folds)} {fold['fold']}: {result.height:,} rows in {seconds / 60:.1f} min, "
              f"{elapsed:.1f} min so far", flush=True)

1/8 2021-Q3: already done, loaded from 2021-Q3.parquet
2/8 2021-Q4: already done, loaded from 2021-Q4.parquet
3/8 2022-Q1: already done, loaded from 2022-Q1.parquet
4/8 2022-Q2: already done, loaded from 2022-Q2.parquet
5/8 2022-Q3: already done, loaded from 2022-Q3.parquet
6/8 2022-Q4: already done, loaded from 2022-Q4.parquet
7/8 2023-Q1: already done, loaded from 2023-Q1.parquet
8/8 2023-Q2: already done, loaded from 2023-Q2.parquet
[perf] 10.backtest: 0.00s | 0.0 cores | peak RAM 1.659 GB | GPU mem 0 MB


In [5]:
fold_frames = []
for fold in folds:
    fold_frames.append(pl.read_parquet(BACKTEST_DIR / f"{fold['fold']}.parquet"))

is_workday = (pl.col("day_of_week") <= 5) & ~pl.col("is_holiday")
segment = (
    pl.when(~is_workday).then(pl.lit("weekend / holiday"))
    .when(pl.col("hour").is_between(6, 9)).then(pl.lit("AM peak (workday 6-9)"))
    .when(pl.col("hour").is_between(15, 18)).then(pl.lit("PM peak (workday 15-18)"))
    .otherwise(pl.lit("workday off-peak"))
)

backtest = pl.concat(fold_frames).with_columns(is_workday.alias("is_workday"), segment.alias("segment"))
print(f"out-of-sample forecasts: {backtest.height:,} station-hours, "
      f"{backtest['date'].min()} to {backtest['date'].max()}")

out-of-sample forecasts: 805,200 station-hours, 2021-07-01 to 2023-05-02


## 3. ผลทดสอบย้อนหลัง

ตัวชี้วัดเหมือน M1
- WAPE = ผลรวม |จริง - พยากรณ์| / ผลรวมค่าจริง (ยิ่งต่ำยิ่งดี)
- baseline คือค่าเฉลี่ย 4 สัปดาห์ของชั่วโมงเดียวกัน (วิธีที่ดีที่สุดที่ไม่ใช้โมเดล)
- coverage = สัดส่วนค่าจริงที่อยู่ในช่วง P10-P90 (เป้าคือ 80%)

In [6]:
def wape(frame, prediction):
    rows = frame.filter(pl.col(prediction).is_not_null())
    error = float((rows["target"] - rows[prediction]).abs().sum())
    total = float(rows["target"].sum())
    return 100 * error / total


def coverage(frame, low="p10", high="p90"):
    inside = frame.filter((pl.col("target") >= pl.col(low)) & (pl.col("target") <= pl.col(high))).height
    return 100 * inside / frame.height


def station_day_wape(frame, prediction):
    daily = frame.group_by("station", "date").agg(pl.col("target").sum(), pl.col(prediction).sum())
    return wape(daily, prediction)


fold_rows = []
for fold in folds:
    rows = backtest.filter(pl.col("fold") == fold["fold"])
    model_wape = wape(rows, "prediction")
    baseline_wape = wape(rows, "lag_4w_mean")
    fold_rows.append({
        "fold": fold["fold"],
        "M1 WAPE": round(model_wape, 2),
        "baseline WAPE": round(baseline_wape, 2),
        "error cut %": round(100 * (baseline_wape - model_wape) / baseline_wape, 1),
        "station-day M1": round(station_day_wape(rows, "prediction"), 2),
        "station-day baseline": round(station_day_wape(rows, "lag_4w_mean"), 2),
        "P10-P90 coverage": round(coverage(rows), 1),
    })

overall_model = wape(backtest, "prediction")
overall_baseline = wape(backtest, "lag_4w_mean")
fold_rows.append({
    "fold": "all 8 quarters",
    "M1 WAPE": round(overall_model, 2),
    "baseline WAPE": round(overall_baseline, 2),
    "error cut %": round(100 * (overall_baseline - overall_model) / overall_baseline, 1),
    "station-day M1": round(station_day_wape(backtest, "prediction"), 2),
    "station-day baseline": round(station_day_wape(backtest, "lag_4w_mean"), 2),
    "P10-P90 coverage": round(coverage(backtest), 1),
})

fold_results = pd.DataFrame(fold_rows).set_index("fold")
fold_results.to_csv(TAB / "m3_backtest_by_quarter.csv")

wins = 0
for fold in folds:
    if fold_results.loc[fold["fold"], "M1 WAPE"] < fold_results.loc[fold["fold"], "baseline WAPE"]: # type: ignore
        wins = wins + 1
print(f"M1 beats the baseline in {wins} of {len(folds)} quarters")
fold_results

M1 beats the baseline in 8 of 8 quarters


,M1 WAPE,baseline WAPE,error cut %,station-day M1,station-day baseline,P10-P90 coverage
fold,,,,,,
2021-Q3,14.31,15.91,10.1,6.80,9.59,80.3
2021-Q4,16.50,20.62,20.0,9.73,14.45,79.4
2022-Q1,15.09,17.83,15.4,10.23,13.51,80.0
2022-Q2,14.94,15.14,1.3,8.11,8.42,80.7
2022-Q3,13.41,16.12,16.8,6.60,9.50,82.2
2022-Q4,13.94,19.00,26.6,7.56,12.73,81.2
2023-Q1,15.32,18.79,18.5,9.91,13.74,78.2
2023-Q2,12.16,12.82,5.2,6.42,7.27,82.5
all 8 quarters,14.57,17.32,15.9,8.24,11.37,80.4


In [7]:
segment_order = ["AM peak (workday 6-9)", "PM peak (workday 15-18)", "workday off-peak", "weekend / holiday"]

segment_rows = []
for fold in folds:
    rows = backtest.filter(pl.col("fold") == fold["fold"])
    values = {"fold": fold["fold"]}
    for name in segment_order:
        part = rows.filter(pl.col("segment") == name)
        values[f"{name}: M1"] = round(wape(part, "prediction"), 1)
        values[f"{name}: baseline"] = round(wape(part, "lag_4w_mean"), 1)
    segment_rows.append(values)

pd.DataFrame(segment_rows).set_index("fold").T

fold,2021-Q3,2021-Q4,2022-Q1,2022-Q2,2022-Q3,2022-Q4,2023-Q1,2023-Q2
AM peak (workday 6-9): M1,9.1,12.1,13.0,9.7,9.4,10.9,13.5,8.7
AM peak (workday 6-9): baseline,11.1,16.0,15.1,9.8,11.0,15.2,17.1,9.3
PM peak (workday 15-18): M1,11.7,13.0,12.5,11.5,9.8,9.9,13.8,8.9
PM peak (workday 15-18): baseline,12.0,14.4,14.5,10.3,10.8,12.9,15.6,9.2
workday off-peak: M1,16.1,17.2,16.2,17.8,15.1,15.4,16.3,14.9
workday off-peak: baseline,16.5,17.5,18.2,17.0,16.3,17.2,18.6,16.0
weekend / holiday: M1,21.8,25.5,20.1,22.6,21.4,21.4,19.2,18.0
weekend / holiday: baseline,26.6,39.2,25.9,26.7,30.7,35.4,27.1,18.6


In [8]:
labels = []
model_values = []
baseline_values = []
for fold in folds:
    labels.append(fold["fold"])
    model_values.append(fold_results.loc[fold["fold"], "M1 WAPE"])
    baseline_values.append(fold_results.loc[fold["fold"], "baseline WAPE"])

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=labels, y=baseline_values, mode="lines+markers", name="baseline: mean of 4 weeks",
    line=dict(color=viz.MUTED, width=2), marker=dict(size=8),
    hovertemplate="%{x}: WAPE %{y:.1f}%<extra>baseline</extra>",
))
fig.add_trace(go.Scatter(
    x=labels, y=model_values, mode="lines+markers", name="M1 retrained each quarter",
    line=dict(color=viz.ACCENT, width=2), marker=dict(size=8),
    hovertemplate="%{x}: WAPE %{y:.1f}%<extra>M1</extra>",
))
fig.update_xaxes(title_text="quarter forecast (model trained only on data before it)")
fig.update_yaxes(title_text="WAPE, station-hour entries (lower is better)", ticksuffix="%", rangemode="tozero")

title = f"M1 beats the baseline in {wins} of {len(folds)} quarters, not just on the 2023 test"
subtitle = f"Rolling backtest Jul 2021 - May 2023: overall WAPE {overall_model:.1f}% vs {overall_baseline:.1f}%, same settings every quarter"
viz.titled(fig, title, subtitle)
viz.save(fig, "10_backtest_wape")
fig

## 4. ขยายช่วง P10-P90 ให้ครอบค่าจริงได้ตามเป้า (conformal calibration)

ปัญหา: ช่วง P10-P90 ของโมเดล quantile แคบเกินไป โดยเฉพาะช่วงเร่งด่วน

วิธี (conformalized quantile regression แบบปรับตามความกว้างของช่วง)
1. ดูไตรมาสก่อนหน้าทั้งหมดที่พยากรณ์ไปแล้ว (ข้อมูลที่รู้คำตอบแล้วจริงในวันนั้น)
2. แต่ละชั่วโมงคิดว่าค่าจริงหลุดออกนอกช่วงไปกี่เท่าของความกว้างช่วง: score = max(P10 - จริง, จริง - P90) / (P90 - P10 + 1)
3. หาค่า score ที่ 80% ของชั่วโมงไม่เกิน (q) แยกตาม 4 ช่วงเวลา (เช้า, เย็น, วันทำงานนอกช่วงเร่งด่วน, วันหยุด)
4. ไตรมาสถัดไปขยายช่วงเป็น P10 - q × กว้าง และ P90 + q × กว้าง (ถ้า q ติดลบ ช่วงจะแคบลง)

ไตรมาสแรก (2021-Q3) ไม่มีไตรมาสก่อนหน้า จึงใช้ช่วงเดิม
ใช้แค่ข้อมูลในอดีตปรับ ไม่ได้ใช้คำตอบของไตรมาสที่วัดผล

In [9]:
TARGET_COVERAGE = 0.80

scored = backtest.with_columns(
    (pl.col("p90") - pl.col("p10") + 1).alias("width"),
).with_columns(
    (pl.max_horizontal(pl.col("p10") - pl.col("target"), pl.col("target") - pl.col("p90")) / pl.col("width")).alias("score"),
)


def calibration_offsets(history):
    offsets = {}
    for name in segment_order:
        scores = history.filter(pl.col("segment") == name)["score"].to_numpy()
        count = len(scores)
        level = min(1.0, np.ceil((count + 1) * TARGET_COVERAGE) / count)
        offsets[name] = float(np.quantile(scores, level))
    return offsets


calibrated_frames = []
offset_rows = []
for position, fold in enumerate(folds):
    rows = scored.filter(pl.col("fold") == fold["fold"])
    if position == 0:
        rows = rows.with_columns(pl.col("p10").cast(pl.Float64).alias("p10_cal"), pl.col("p90").cast(pl.Float64).alias("p90_cal"))
        calibrated_frames.append(rows)
        continue

    earlier = []
    for previous in folds[:position]:
        earlier.append(previous["fold"])
    offsets = calibration_offsets(scored.filter(pl.col("fold").is_in(earlier)))

    offset_expression = pl.lit(0.0)
    for name in segment_order:
        offset_expression = pl.when(pl.col("segment") == name).then(pl.lit(offsets[name])).otherwise(offset_expression)

    rows = rows.with_columns(offset_expression.alias("offset")).with_columns(
        pl.max_horizontal(pl.col("p10") - pl.col("offset") * pl.col("width"), pl.lit(0.0)).alias("p10_cal"),
        (pl.col("p90") + pl.col("offset") * pl.col("width")).alias("p90_cal"),
    ).drop("offset")
    calibrated_frames.append(rows)

    offset_row = {"fold": fold["fold"]}
    for name in segment_order:
        offset_row[name] = round(offsets[name], 3)
    offset_rows.append(offset_row)

calibrated = pl.concat(calibrated_frames, how="diagonal")
print("offset q (in band widths) learned from earlier quarters:")
pd.DataFrame(offset_rows).set_index("fold")

offset q (in band widths) learned from earlier quarters:


,AM peak (workday 6-9),PM peak (workday 15-18),workday off-peak,weekend / holiday
fold,,,,
2021-Q4,0.066,0.059,-0.029,-0.021
2022-Q1,0.099,0.056,-0.034,-0.002
2022-Q2,0.122,0.060,-0.033,-0.013
2022-Q3,0.092,0.063,-0.032,-0.012
2022-Q4,0.082,0.052,-0.035,-0.015
2023-Q1,0.080,0.044,-0.034,-0.016
2023-Q2,0.093,0.052,-0.031,-0.015


In [10]:
evaluated = calibrated.filter(pl.col("fold") != folds[0]["fold"])

coverage_rows = []
for name in ["all hours"] + segment_order:
    if name == "all hours":
        part = evaluated
    else:
        part = evaluated.filter(pl.col("segment") == name)
    raw_width = float((part["p90"] - part["p10"]).mean())
    calibrated_width = float((part["p90_cal"] - part["p10_cal"]).mean())
    coverage_rows.append({
        "segment": name,
        "coverage before %": round(coverage(part), 1),
        "coverage after %": round(coverage(part, "p10_cal", "p90_cal"), 1),
        "band width before": round(raw_width, 1),
        "band width after": round(calibrated_width, 1),
        "width change %": round(100 * (calibrated_width - raw_width) / raw_width, 1),
    })

coverage_table = pd.DataFrame(coverage_rows).set_index("segment")
coverage_table.to_csv(TAB / "m3_calibration.csv")
coverage_table

,coverage before %,coverage after %,band width before,band width after,width change %
segment,,,,,
all hours,80.4,80.0,38.4,39.2,2.1
AM peak (workday 6-9),72.6,79.6,57.5,68.0,18.3
PM peak (workday 15-18),75.8,80.5,71.6,79.6,11.2
workday off-peak,82.8,79.9,27.8,25.7,-7.4
weekend / holiday,81.5,80.2,34.8,33.6,-3.4


In [11]:
by_fold_rows = []
for fold in folds[1:]:
    rows = calibrated.filter(pl.col("fold") == fold["fold"])
    by_fold_rows.append({
        "fold": fold["fold"],
        "before": round(coverage(rows), 1),
        "after": round(coverage(rows, "p10_cal", "p90_cal"), 1),
        "AM peak before": round(coverage(rows.filter(pl.col("segment") == segment_order[0])), 1),
        "AM peak after": round(coverage(rows.filter(pl.col("segment") == segment_order[0]), "p10_cal", "p90_cal"), 1),
    })
coverage_by_fold = pd.DataFrame(by_fold_rows).set_index("fold")
coverage_by_fold

,before,after,AM peak before,AM peak after
fold,,,,
2021-Q4,79.4,78.7,69.4,74.9
2022-Q1,80.0,80.3,66.5,75.4
2022-Q2,80.7,80.5,79.3,87.3
2022-Q3,82.2,81.9,77.2,83.6
2022-Q4,81.2,80.5,74.1,80.9
2023-Q1,78.2,77.5,67.1,73.3
2023-Q2,82.5,82.1,77.7,84.6


In [12]:
fig = go.Figure()
series = [
    ("AM peak before", "AM peak, model band", viz.CONTEXT, "dot"),
    ("AM peak after", "AM peak, calibrated band", viz.CONTRAST, "solid"),
    ("before", "all hours, model band", viz.MUTED, "dot"),
    ("after", "all hours, calibrated band", viz.ACCENT, "solid"),
]
for column, label, color, dash in series:
    fig.add_trace(go.Scatter(
        x=list(coverage_by_fold.index), y=coverage_by_fold[column], mode="lines+markers", name=label,
        line=dict(color=color, width=2, dash=dash), marker=dict(size=8),
        hovertemplate="%{x}: %{y:.1f}% inside<extra>" + label + "</extra>",
    ))
fig.add_hline(y=80, line=dict(color=viz.INK2, width=1, dash="dash"))
fig.add_annotation(x=1, xref="paper", y=80, text="target 80%", showarrow=False, xanchor="right", yshift=10,
                   font=dict(size=11, color=viz.INK2))
fig.update_xaxes(title_text="quarter (calibrated with earlier quarters only)")
fig.update_yaxes(title_text="% of actual values inside P10-P90", ticksuffix="%", range=[50, 100])

after_all = coverage_table.loc["all hours", "coverage after %"]
after_am = coverage_table.loc[segment_order[0], "coverage after %"]
before_am = coverage_table.loc[segment_order[0], "coverage before %"]
title = f"Calibration lifts AM-peak coverage from {before_am:.0f}% to {after_am:.0f}%"
subtitle = f"Share of station-hours inside the P10-P90 band, 2021-Q4 to 2023-Q2; all hours {after_all:.0f}% after calibration"
viz.titled(fig, title, subtitle)
viz.save(fig, "10_calibration")
fig

## 5. ตรวจจับเหตุการณ์ผิดปกติ

ใช้ค่าพยากรณ์ล่วงหน้าจากข้อ 2 กับช่วงที่ปรับแล้วจากข้อ 4 แล้วสรุปเป็นระดับสถานี-วัน
- **คนเกินช่วง** = ผลรวมของ (จริง - P90) ในชั่วโมงที่ค่าจริงสูงกว่า P90 คือจำนวนคนที่เกินแผนสำรองไปแล้ว
- **คนขาดช่วง** = ผลรวมของ (P10 - จริง) ในชั่วโมงที่ค่าจริงต่ำกว่า P10

เกณฑ์ว่าเป็นเหตุการณ์ ต้องผ่านทั้งสองข้อ
- คนเกิน (หรือขาด) ช่วงรวมทั้งวันอย่างน้อย 25% ของค่าพยากรณ์ P50 ของวันนั้น จึงใหญ่พอจะกระทบการเดินรถ
- และอย่างน้อย 300 คน กันไม่ให้สถานีเล็กที่แกว่งไม่กี่สิบคนถูกนับ

ใช้ทุกวันใน 8 ไตรมาส ยกเว้นไตรมาสแรกที่ยังไม่มีช่วงที่ปรับแล้ว (ใช้ช่วงเดิมของโมเดล)

In [13]:
EVENT_SHARE = 0.25
EVENT_MIN_RIDERS = 300

station_days = calibrated.group_by("station", "date").agg(
    pl.col("target").sum().alias("actual"),
    pl.col("p50").sum().alias("p50"),
    (pl.col("target") - pl.col("p90_cal")).clip(lower_bound=0).sum().alias("over"),
    (pl.col("p10_cal") - pl.col("target")).clip(lower_bound=0).sum().alias("under"),
    (pl.col("target") > pl.col("p90_cal")).sum().alias("hours_over"),
    (pl.col("target") < pl.col("p10_cal")).sum().alias("hours_under"),
    pl.col("lag_4w_mean").sum().alias("reference_4w"),
    pl.col("is_workday").first(),
    pl.col("is_holiday").first(),
).with_columns(
    ((pl.col("over") >= EVENT_SHARE * pl.col("p50")) & (pl.col("over") >= EVENT_MIN_RIDERS)).alias("surge"),
    ((pl.col("under") >= EVENT_SHARE * pl.col("p50")) & (pl.col("under") >= EVENT_MIN_RIDERS)).alias("drop"),
)

surge_count = int(station_days["surge"].sum())
drop_count = int(station_days["drop"].sum())
print(f"station-days checked: {station_days.height:,}")
print(f"surge station-days  : {surge_count} ({100 * surge_count / station_days.height:.2f}%)")
print(f"drop station-days   : {drop_count} ({100 * drop_count / station_days.height:.2f}%)")

station-days checked: 33,550
surge station-days  : 281 (0.84%)
drop station-days   : 46 (0.14%)


In [14]:
sensitivity_rows = []
for share in [0.15, 0.20, 0.25, 0.30, 0.35]:
    for floor in [200, 300, 500]:
        surge_flag = (pl.col("over") >= share * pl.col("p50")) & (pl.col("over") >= floor)
        drop_flag = (pl.col("under") >= share * pl.col("p50")) & (pl.col("under") >= floor)
        flagged = station_days.with_columns(surge_flag.alias("s"), drop_flag.alias("d"))
        by_station = flagged.group_by("station").agg(pl.col("s").sum()).sort("s", descending=True)
        sensitivity_rows.append({
            "share of P50": share,
            "min riders": floor,
            "surge days": int(flagged["s"].sum()),
            "drop days": int(flagged["d"].sum()),
            "top surge station": by_station["station"][0],
            "its share of surges %": round(100 * by_station["s"][0] / flagged["s"].sum(), 1),
        })

pd.DataFrame(sensitivity_rows)

,share of P50,min riders,surge days,drop days,top surge station,its share of surges %
0,0.15,200,528,126,COLS,23.3
1,0.15,300,422,86,COLS,27.7
2,0.15,500,297,50,COLS,32.7
3,0.20,200,405,84,COLS,27.2
4,0.20,300,333,59,COLS,31.8
5,0.20,500,262,34,COLS,37.0
6,0.25,200,333,66,COLS,31.8
7,0.25,300,281,46,COLS,37.0
8,0.25,500,234,26,COLS,41.5
9,0.30,200,274,50,COLS,35.0


เปลี่ยนเกณฑ์แล้วจำนวนเหตุการณ์เปลี่ยนตาม แต่สถานีที่มีคนล้นบ่อยที่สุดยังเป็น COLS ทุกเกณฑ์ ข้อสรุปหลักจึงไม่ได้ขึ้นกับเกณฑ์ที่เลือก

### 5.1 วันที่เกิดพร้อมกันหลายสถานี (ระดับทั้งระบบ)

ถ้าหลายสถานีผิดปกติในวันเดียวกัน สาเหตุมักเป็นเรื่องระดับเมือง เช่น งานใหญ่ พายุ หรือวันหยุด
ตารางนี้ดูวันที่มีอย่างน้อย 5 สถานี พร้อมเทียบระดับทั้งระบบ
- actual / P50 = คนจริงทั้งระบบเทียบกับค่าพยากรณ์
- reference / actual = ค่าเฉลี่ย 4 สัปดาห์ก่อน (ฐานของ lag) เทียบกับคนจริง ถ้าต่ำกว่า 1 มาก แปลว่าสัปดาห์อ้างอิงเองผิดปกติ (มีวันหยุดหรือพายุ) โมเดลจึงพยากรณ์ต่ำตาม

คอลัมน์ "likely cause" มาจากความรู้ภายนอกเกี่ยวกับเหตุการณ์ในเขต Bay Area ไม่ได้มาจากข้อมูลชุดนี้ วันที่ไม่แน่ใจเขียนว่า not verified

In [15]:
KNOWN_CAUSES = {
    date(2021, 10, 24): "atmospheric river storm (external)",
    date(2021, 12, 24): "observed Christmas holiday, travel day busier than a holiday",
    date(2022, 6, 20): "Warriors championship parade in SF (external)",
    date(2022, 6, 26): "SF Pride, first full parade since 2019 (external)",
    date(2023, 1, 4): "atmospheric river storm (external)",
}

system_days = calibrated.group_by("date").agg(
    pl.col("target").sum().alias("actual"),
    pl.col("p50").sum().alias("p50"),
    pl.col("lag_4w_mean").sum().alias("reference_4w"),
    pl.col("day_of_week").first(),
)
flag_counts = station_days.group_by("date").agg(
    pl.col("surge").sum().alias("surge_stations"),
    pl.col("drop").sum().alias("drop_stations"),
)
system_days = system_days.join(flag_counts, on="date").with_columns(
    (pl.col("actual") / pl.col("p50")).alias("actual_vs_p50"),
    (pl.col("reference_4w") / pl.col("actual")).alias("reference_vs_actual"),
    (pl.col("surge_stations") + pl.col("drop_stations")).alias("stations_flagged"),
)

wide_days = system_days.filter(pl.col("stations_flagged") >= 5).sort("stations_flagged", descending=True)

day_names = ["", "Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
wide_rows = []
for row in wide_days.to_dicts():
    cause = "not verified"
    if row["date"] in KNOWN_CAUSES:
        cause = KNOWN_CAUSES[row["date"]]
    elif row["reference_vs_actual"] < 0.8:
        cause = "reference weeks were abnormal (holidays / storms)"
    wide_rows.append({
        "date": row["date"],
        "day": day_names[row["day_of_week"]],
        "surge stations": row["surge_stations"],
        "drop stations": row["drop_stations"],
        "actual / P50": round(row["actual_vs_p50"], 2),
        "reference / actual": round(row["reference_vs_actual"], 2),
        "likely cause": cause,
    })

wide_table = pd.DataFrame(wide_rows)
wide_table.to_csv(TAB / "m3_system_events.csv", index=False)
wide_table

,date,day,surge stations,drop stations,actual / P50,reference / actual,likely cause
0,2022-06-20,Mon,35,0,1.48,0.56,Warriors championship parade in SF (external)
1,2022-06-26,Sun,13,0,1.80,0.46,"SF Pride, first full parade since 2019 (external)"
2,2021-10-24,Sun,0,10,0.58,1.75,atmospheric river storm (external)
3,2023-01-23,Mon,8,0,1.42,0.56,reference weeks were abnormal (holidays / storms)
4,2021-12-24,Fri,8,0,1.66,2.18,"observed Christmas holiday, travel day busier ..."
5,2023-01-18,Wed,7,0,1.31,0.73,reference weeks were abnormal (holidays / storms)
6,2023-01-21,Sat,6,0,1.34,0.69,reference weeks were abnormal (holidays / storms)


In [16]:
timeline = system_days.sort("date")

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=timeline["date"], y=timeline["actual_vs_p50"], mode="lines", name="actual / P50, whole system",
    line=dict(color=viz.CONTEXT, width=1.5),
    hovertemplate="%{x|%a %d %b %Y}: %{y:.2f}<extra></extra>",
))

marked = timeline.filter(pl.col("stations_flagged") >= 5)
fig.add_trace(go.Scatter(
    x=marked["date"], y=marked["actual_vs_p50"], mode="markers", name="5 or more stations flagged",
    marker=dict(size=9, color=viz.CONTRAST, line=dict(color="white", width=1)),
    customdata=marked["stations_flagged"],
    hovertemplate="%{x|%a %d %b %Y}: %{y:.2f}, %{customdata} stations<extra></extra>",
))

for event_day, label in [(date(2022, 6, 20), "Warriors parade"), (date(2022, 6, 26), "Pride"),
                         (date(2021, 10, 24), "storm"), (date(2023, 1, 4), "storm"),
                         (date(2021, 12, 24), "Christmas Eve (observed holiday)"),
                         (date(2023, 1, 23), "after holiday and storm weeks")]:
    value = timeline.filter(pl.col("date") == event_day)["actual_vs_p50"][0]
    if value > 1:
        shift = -30
    else:
        shift = 30
    fig.add_annotation(x=event_day, y=value, text=label, showarrow=True, arrowhead=0, arrowcolor=viz.MUTED,
                       ax=0, ay=shift, font=dict(size=11, color=viz.INK))

fig.add_hline(y=1, line=dict(color=viz.MUTED, width=1, dash="dot"))
fig.update_layout(hovermode="closest")
fig.update_yaxes(title_text="actual entries / forecast P50 (1 = on forecast)")

title = "Most days land close to the forecast; the big misses are city events, storms and holiday weeks"
subtitle = "Whole-system daily entries vs the out-of-sample P50 forecast, Jul 2021 - May 2023"
viz.titled(fig, title, subtitle)
viz.save(fig, "10_system_events")
fig

### 5.2 เหตุการณ์เฉพาะสถานี

ตัดวันที่เกิดพร้อมกันตั้งแต่ 5 สถานีขึ้นไปออก (เป็นเรื่องระดับเมืองจากข้อ 5.1) เหลือเหตุการณ์ที่เกิดเฉพาะสถานี

In [17]:
wide_dates = wide_days["date"].to_list()
local = station_days.filter(~pl.col("date").is_in(wide_dates))

per_station = local.group_by("station").agg(
    pl.col("surge").sum().alias("surge_days"),
    pl.col("drop").sum().alias("drop_days"),
    pl.col("over").filter(pl.col("surge")).sum().alias("surge_riders"),
).filter((pl.col("surge_days") > 0) | (pl.col("drop_days") > 0)).sort("surge_days", descending=True)

print(f"station-specific surge days: {int(local['surge'].sum())}, drop days: {int(local['drop'].sum())}")
per_station.head(15).to_pandas()

station-specific surge days: 204, drop days: 36


,station,surge_days,drop_days,surge_riders
0,COLS,103,0,139744.099268
1,EMBR,18,1,39810.095340
2,DBRK,16,0,15694.407267
3,CIVC,8,1,12941.718139
4,DALY,8,0,6881.260889
5,DUBL,8,0,3201.792644
6,MONT,8,0,14759.331238
7,MLPT,6,0,2694.745390
8,19TH,5,0,3422.529728
9,OAKL,4,0,1934.444029


In [18]:
top_stations = per_station.head(12).sort("surge_days")

fig = go.Figure(go.Bar(
    x=top_stations["surge_days"], y=top_stations["station"], orientation="h", marker_color=viz.ACCENT,
    text=top_stations["surge_days"], textposition="outside",
    hovertemplate="%{y}: %{x} surge days<extra></extra>",
))
fig.update_layout(hovermode="closest", showlegend=False, margin=dict(l=80))
fig.update_xaxes(title_text="station-specific surge days, Jul 2021 - May 2023", showgrid=True, gridcolor=viz.GRID)
fig.update_yaxes(showgrid=False)

cols_days = int(per_station.filter(pl.col("station") == "COLS")["surge_days"][0])
title = f"Coliseum (COLS) had {cols_days} surge days, more than any other station"
subtitle = "Days when riders above the calibrated P90 reached 25% of the forecast and 300 people; city-wide days removed"
viz.titled(fig, title, subtitle)
viz.save(fig, "10_event_stations", height=480)
fig

### 5.3 ตรวจว่าเหตุการณ์ที่ COLS เป็นของจริง

COLS อยู่ข้าง Oakland Coliseum ซึ่งใช้จัดเกมเบสบอล (ฤดูกาลประมาณ เม.ย. ถึง ต.ค.) และคอนเสิร์ต
ถ้าสิ่งที่จับได้เป็นงานที่สนามจริง ควรเห็น 3 อย่างในข้อมูล
1. เกิดในเดือน เม.ย.-ต.ค. มากกว่าสัดส่วนของวันในช่วงนั้น
2. เกิดวันหยุดสุดสัปดาห์มากกว่าวันทำงาน (เทียบกับสัดส่วนวัน)
3. คนเกินช่วงกระจุกอยู่ในช่วงเย็นถึงดึก ตอนคนเข้าสถานีกลับบ้านหลังจบงาน

In [19]:
cols_all = local.filter(pl.col("station") == "COLS").with_columns(
    pl.col("date").dt.month().is_between(4, 10).alias("season"),
)
cols_events = cols_all.filter(pl.col("surge"))

season_share_days = 100 * cols_all["season"].mean()
season_share_events = 100 * cols_events["season"].mean()
weekend_share_days = 100 * (~cols_all["is_workday"]).mean()
weekend_share_events = 100 * (~cols_events["is_workday"]).mean()

cols_hours = calibrated.filter(pl.col("station") == "COLS").join(
    cols_events.select("date"), on="date", how="semi",
).with_columns((pl.col("target") - pl.col("p90_cal")).clip(lower_bound=0).alias("over"))
evening_share = 100 * cols_hours.filter(pl.col("hour") >= 17)["over"].sum() / cols_hours["over"].sum()

check = pd.DataFrame([
    {"check": "share in Apr-Oct", "all COLS days %": round(season_share_days, 1), "COLS surge days %": round(season_share_events, 1)},
    {"check": "share on weekends / holidays", "all COLS days %": round(weekend_share_days, 1), "COLS surge days %": round(weekend_share_events, 1)},
]).set_index("check")
print(f"share of the extra riders entering from 17:00 onwards: {evening_share:.1f}%")
check

share of the extra riders entering from 17:00 onwards: 61.4%


,all COLS days %,COLS surge days %
check,,
share in Apr-Oct,55.1,86.4
share on weekends / holidays,30.9,53.4


In [20]:
month_counts = cols_events.with_columns(pl.col("date").dt.month().alias("month")).group_by("month").len().sort("month")
month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

values = []
colors = []
for month in range(1, 13):
    found = month_counts.filter(pl.col("month") == month)
    if found.height == 0:
        values.append(0)
    else:
        values.append(int(found["len"][0]))
    if 4 <= month <= 10:
        colors.append(viz.ACCENT)
    else:
        colors.append(viz.CONTEXT)

fig = go.Figure(go.Bar(
    x=month_names, y=values, marker_color=colors, text=values, textposition="outside",
    hovertemplate="%{x}: %{y} surge days<extra></extra>",
))
fig.update_layout(hovermode="closest", showlegend=False)
fig.update_yaxes(title_text="COLS surge days (Jul 2021 - May 2023)", rangemode="tozero")

title = f"{season_share_events:.0f}% of Coliseum surges fall in the Apr-Oct baseball season"
subtitle = f"Blue = Apr-Oct; {evening_share:.0f}% of the extra riders enter from 17:00 onwards, when events end"
viz.titled(fig, title, subtitle)
viz.save(fig, "10_cols_season", height=460)
fig

### 5.4 วันที่คนหายเฉพาะสถานี

คนหายเฉพาะสถานี (ไม่ใช่ทั้งเมือง) มักมาจากสถานีปิดบางส่วน ระบบขัดข้องเฉพาะจุด หรือข้อมูลประตูกั้นขาด ต้องให้ฝ่ายปฏิบัติการตรวจสอบสาเหตุ

In [21]:
drops = local.filter(pl.col("drop")).with_columns(
    pl.col("p50").round(0),
    (pl.col("actual") / pl.col("p50")).round(2).alias("actual_vs_p50"),
).sort("under", descending=True)

drop_dates = drops.group_by("station").agg(pl.len().alias("drop_days"), pl.col("date").min().alias("first"), pl.col("date").max().alias("last"))
print(drop_dates.sort("drop_days", descending=True).head(5))
drops.select("station", "date", "actual", "p50", "actual_vs_p50", "hours_under").head(15).to_pandas()

shape: (5, 4)
┌─────────┬───────────┬────────────┬────────────┐
│ station ┆ drop_days ┆ first      ┆ last       │
│ ---     ┆ ---       ┆ ---        ┆ ---        │
│ str     ┆ u32       ┆ date       ┆ date       │
╞═════════╪═══════════╪════════════╪════════════╡
│ MLBR    ┆ 14        ┆ 2021-10-25 ┆ 2023-01-04 │
│ CONC    ┆ 5         ┆ 2022-07-19 ┆ 2022-10-30 │
│ ANTC    ┆ 3         ┆ 2021-12-29 ┆ 2022-11-08 │
│ PITT    ┆ 2         ┆ 2022-07-19 ┆ 2022-07-22 │
│ RICH    ┆ 2         ┆ 2021-09-23 ┆ 2022-03-07 │
└─────────┴───────────┴────────────┴────────────┘


,station,date,actual,p50,actual_vs_p50,hours_under
0,EMBR,2023-01-04,6409.0,9447.0,0.68,11
1,POWL,2023-01-04,6750.0,10481.0,0.64,17
2,CIVC,2021-12-21,4027.0,6825.0,0.59,10
3,MLBR,2022-12-01,27.0,2129.0,0.01,19
4,DELN,2022-03-07,1312.0,2689.0,0.49,7
5,MLBR,2021-12-22,2.0,1333.0,0.00,20
6,ANTC,2022-11-08,926.0,2061.0,0.45,10
7,MLBR,2021-12-13,45.0,1168.0,0.04,19
8,MLBR,2022-11-08,747.0,2091.0,0.36,18
9,CONC,2022-07-19,1222.0,2128.0,0.57,9


## เก็บผลไว้ใช้ต่อ

เซฟค่าพยากรณ์รายสถานี-ชั่วโมงที่ปรับช่วงแล้ว และผลรายสถานี-วัน (รวมธงวันผิดปกติ) ไว้ให้ notebook 12 ใช้ทำตัวสำรวจผลพยากรณ์ โดยไม่ต้องคำนวณซ้ำ

In [22]:
calibrated.select(
    "station", "date", "hour", "fold", "segment", "target", "p10", "p50", "p90", "p10_cal", "p90_cal",
).write_parquet(MODELS / "m3_calibrated.parquet")
station_days.write_parquet(MODELS / "m3_station_days.parquet")
print(f"saved m3_calibrated.parquet ({calibrated.height:,} rows) and m3_station_days.parquet ({station_days.height:,} rows)")

saved m3_calibrated.parquet (805,200 rows) and m3_station_days.parquet (33,550 rows)


## สรุป: Insight → Pain point

pain point ของ BART ในข้อนี้คือ ต้องมั่นใจก่อนใช้โมเดลวางแผนจริง ช่วง P90 ต้องเชื่อถือได้สำหรับขบวนสำรอง และต้องรู้ล่วงหน้าว่าวันไหนสถานีไหนจะผิดปกติ

| # | Insight (ตัวเลข) | Pain point | ตอบโจทย์แค่ไหน | ข้อเสนอ |
|---|---|---|---|---|
| 1 | ทดสอบย้อนหลัง 8 ไตรมาส (ก.ค. 2021 ถึง พ.ค. 2023) เทรนใหม่ทุกไตรมาสด้วยค่าเดิมของ M1 ชนะ baseline ครบ 8 ไตรมาส WAPE รวม 14.6% เทียบกับ 17.3% (ลด error 16%) ระดับสถานี-วัน 8.2% เทียบกับ 11.4% | โมเดลเชื่อถือได้จริงไหม | ตรงจุด ผลบน test ของ M1 (14.5%) ไม่ได้บังเอิญ | เทรนใหม่ทุกไตรมาสด้วยค่าเดิมได้ ใช้เวลา ~4 นาทีต่อครั้งบน GPU ไม่ต้องจูนใหม่ |
| 2 | ชนะมากที่สุดในวันหยุด/สุดสัปดาห์ เช่น 2021-Q4 25.5% เทียบกับ 39.2% และ 2022-Q4 21.4% เทียบกับ 35.4% แต่บางไตรมาสที่ระดับผู้โดยสารนิ่ง ชนะน้อย (2022-Q2 14.9% เทียบกับ 15.1%) | วันที่แผนเดิมพลาดบ่อย | ตรงจุด | ใช้ M1 กับไตรมาสที่มีวันหยุดยาวเป็นพิเศษ ส่วนช่วงที่นิ่ง ค่าเฉลี่ย 4 สัปดาห์ก็ใกล้เคียง |
| 3 | ขยายช่วงด้วย error ของไตรมาสก่อน ทำให้ช่วงเช้าครอบค่าจริงจาก 72.6% เป็น 79.6% และเย็นจาก 75.8% เป็น 80.5% โดยรวม 80.0% ตรงเป้า ช่วงเช้ากว้างขึ้น 18% แต่ช่วงนอกเร่งด่วนแคบลง 7% ความกว้างรวมเพิ่มแค่ 2% | ขบวนสำรอง (P90) ช่วงเร่งด่วน | ตรงจุด แก้ข้อจำกัดของ M1 แต่รายไตรมาสยังแกว่ง 73-87% ในช่วงเช้า | ใช้ P90 ที่ปรับแล้วกำหนดขบวนสำรอง ได้ย้ายความจุสำรองจากช่วงนอกเร่งด่วนไปช่วงเร่งด่วน โดยแทบไม่เพิ่มรวม |
| 4 | วันที่พลาดมากที่สุดเป็นเหตุการณ์ระดับเมืองที่โมเดลไม่มีทางรู้ เช่น 20 มิ.ย. 2022 (พาเหรด Warriors) คนทั้งระบบสูงกว่าพยากรณ์ 48% ใน 35 สถานี, Pride 26 มิ.ย. 2022 สูงกว่า 80%, พายุ 24 ต.ค. 2021 ต่ำกว่า 42% ใน 10 สถานี | เตรียมรถและพนักงานวันพิเศษ | บางส่วน จับได้หลังเกิด แต่พยากรณ์ล่วงหน้าไม่ได้ถ้าไม่มีข้อมูลงาน | ทำปฏิทินงานใหญ่ของเมือง (พาเหรด, Pride, คอนเสิร์ต) แล้วปรับแผนด้วยคนก่อนใช้ค่าพยากรณ์ |
| 5 | COLS มีวันคนล้นเฉพาะสถานี 103 วัน (ครึ่งหนึ่งของทั้งหมด 204) 86% อยู่ใน เม.ย.-ต.ค. (วันทั้งหมดอยู่ในช่วงนี้แค่ 55%) 53% เป็นวันหยุด (ปกติ 31%) และ 61% ของคนที่เกินเข้าสถานีหลัง 17:00 ตรงกับงานที่สนามกีฬา | ขบวนหลังงานกีฬา | ตรงจุด ตรวจกับรูปแบบในข้อมูลแล้วสอดคล้อง 3 ข้อ และ COLS ยังอันดับหนึ่งทุกเกณฑ์ | นำตารางการแข่งขันและงานที่ Coliseum เข้าโมเดลเป็น feature เป็นจุดที่คุ้มที่สุดในการปรับปรุง M1 ต่อ |
| 6 | ม.ค. 2023 มี 3 วันที่สูงกว่าพยากรณ์ 31-42% แต่ไม่ใช่งาน เพราะ 4 สัปดาห์อ้างอิงมีวันหยุดปีใหม่และพายุ ค่าอ้างอิงจึงต่ำกว่าจริง 27-44% โมเดลเลยพยากรณ์ต่ำตาม | ความแม่นหลังช่วงผิดปกติ | ตรงจุด เป็นจุดอ่อนของ M1 ที่ต้องแก้ | ให้ lag ข้ามวันหยุดและวันพายุ (ใช้ค่าว่างเหมือนวันที่ไม่ใช่ normal) ในรอบปรับปรุงถัดไป |
| 7 | คนหายเฉพาะสถานี 36 วัน MLBR มี 14 วัน คนเข้าเกือบเป็นศูนย์ในวันที่ 12-13 ธ.ค. และ 22-29 ธ.ค. 2021 | ข้อมูลและการปิดสถานี | บางส่วน ข้อมูลบอกได้ว่าผิดปกติ แต่บอกสาเหตุไม่ได้ | ให้ฝ่ายปฏิบัติการตรวจว่าปิดสถานีหรือข้อมูลประตูกั้นขาด และตัดวันเหล่านี้ออกจาก lag |

ทรัพยากรที่ใช้: ทดสอบย้อนหลัง 8 ไตรมาส (เทรน 16 โมเดลบน 4.1-4.9 ล้านแถว) ~32 นาทีบน RTX 4060 ใช้ VRAM ~4.5 GB ส่วนปรับช่วงและตรวจจับเหตุการณ์ไม่กี่วินาทีด้วย Polars

ข้อจำกัด
- ใช้จำนวนต้นไม้คงที่ทุกไตรมาส ไม่มี early stopping รายไตรมาส
- สาเหตุของเหตุการณ์ระดับเมืองมาจากความรู้ภายนอก ไม่ได้มาจากข้อมูลชุดนี้ วันที่ไม่แน่ใจระบุว่า not verified
- เกณฑ์เหตุการณ์ (25% ของ P50 และ 300 คน) เป็นการตัดสินใจ แต่ข้อสรุปหลักไม่เปลี่ยนเมื่อเปลี่ยนเกณฑ์
- ดูแค่ entries ไม่ได้ดูโหลดบนขบวน

ต่อไป: 11_summary